In [5]:
#https://raw.githubusercontent.com/metatron-app/metatron-doc-discovery/refs/heads/master/_static/data/sales-data-sample.csv


from pathlib import Path
import pandas as pd
from urllib.request import urlretrieve

url = "https://raw.githubusercontent.com/metatron-app/metatron-doc-discovery/refs/heads/master/_static/data/sales-data-sample.csv"



df = pd.read_csv(url)
# print(df.head())

print(df.columns)

Index(['OrderDate', 'Category', 'City', 'Country', 'CustomerName', 'Discount',
       'OrderID', 'PostalCode', 'ProductName', 'Profit', 'Quantity', 'Region',
       'Sales', 'Segment', 'ShipDate', 'ShipMode', 'State', 'Sub_Category',
       'DaystoShipActual', 'SalesForecast', 'ShipStatus',
       'DaystoShipScheduled', 'OrderProfitable', 'SalesperCustomer',
       'ProfitRatio', 'SalesaboveTarget', 'latitude', 'longitude'],
      dtype='str')


In [3]:
total_transactions = df["OrderID"].nunique()
total_quantity = df["Quantity"].sum()
total_revenue = df["Sales"].sum()

revenue_by_product = df.groupby("ProductName")["Sales"].sum().sort_values(ascending=False)
revenue_by_category = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
quantity_by_category = df.groupby("Category")["Quantity"].sum().sort_values(ascending=False)
average_product_price = (df["Sales"] / df["Quantity"]).mean()

print("Total number of transactions:", total_transactions)
print("Total quantity sold:", total_quantity)
print("Total revenue:", total_revenue)
print("Product with the highest revenue:", revenue_by_product.index[0])
print("Top 3 products by revenue:")
print(revenue_by_product.head(3))
print("Total revenue by category:")
print(revenue_by_category)
print("Average product price:", round(average_product_price, 2))
print("Total quantity sold by category:")
print(quantity_by_category)

Total number of transactions: 5009
Total quantity sold: 37873
Total revenue: 2297354
Product with the highest revenue: Canon imageCLASS 2200 Advanced Copier
Top 3 products by revenue:
ProductName
Canon imageCLASS 2200 Advanced Copier                                          61600
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind    27454
Cisco TelePresence System EX90 Videoconferencing Unit                          22638
Name: Sales, dtype: int64
Total revenue by category:
Category
Technology         836221
Furniture          742006
Office Supplies    719127
Name: Sales, dtype: int64
Average product price: 60.93
Total quantity sold by category:
Category
Office Supplies    22906
Furniture           8028
Technology          6939
Name: Quantity, dtype: int64


In [17]:
import pandas as pd

data = {
    "age": [22, 25, 28, None, 35, 120, 29, 31],
    "salary": [25000, 30000, 35000, 40000, None, 50000, -1000, 45000],
    "experience": [1, 2, 4, 5, 10, 20, 3, None],
    "city": ["Bangalore", "Mumbai", "Delhi", "Bangalore",
             "Mumbai", None, "Delhi", "Bangalore"],
    "purchased": [0, 1, 1, 0, 1, 1, 0, 1]
}

df = pd.DataFrame(data)
df.loc[~df["age"].between(18, 80), "age"] = pd.NA
df["age"] = df["age"].fillna(df["age"].median())

df.loc[df["salary"] <= 0, "salary"] = pd.NA
df["salary"] = df["salary"].fillna(df["salary"].median())

df["experience"] = df["experience"].fillna(
    df["experience"].median()
)

df["city"] = df["city"].fillna("Unknown")

df


,age,salary,experience,city,purchased
0,22.0,25000.0,1.0,Bangalore,0
1,25.0,30000.0,2.0,Mumbai,1
2,28.0,35000.0,4.0,Delhi,1
3,28.5,40000.0,5.0,Bangalore,0
4,35.0,37500.0,10.0,Mumbai,1
5,28.5,50000.0,20.0,Unknown,1
6,29.0,37500.0,3.0,Delhi,0
7,31.0,45000.0,4.0,Bangalore,1


In [25]:
# Use numpy to calculate these
# Mean age, Maximum salary, Minimum salary, Mean experience

import numpy as np

data = {
    "age": [22, 25, 28, None, 35, 120, 29, 31],
    "salary": [25000, 30000, 35000, 40000, None, 50000, -1000, 45000],
    "experience": [1, 2, 4, 5, 10, 20, 3, None],
    "city": ["Bangalore", "Mumbai", "Delhi", "Bangalore",
             "Mumbai", None, "Delhi", "Bangalore"],
    "purchased": [0, 1, 1, 0, 1, 1, 0, 1]
}

# Normalize the salary:
# normalized_salary = (salary - mean) / std

age = np.array(data["age"], dtype=float)
salary = np.array(data["salary"], dtype=float)
experience = np.array(data["experience"], dtype=float)

age_mean = np.nanmean(age)
salary_max = np.nanmax(salary)
salary_min = np.nanmin(salary)
experience_mean = np.nanmean(experience)

normalized_salary = (salary - np.nanmean(salary)) / np.nanstd(salary)

print("Mean age:", age_mean)
print("Maximum salary:", salary_max)
print("Minimum salary:", salary_min)
print("Mean experience:", experience_mean)
print("Normalized salary:", normalized_salary)



Mean age: 41.42857142857143
Maximum salary: 50000.0
Minimum salary: -1000.0
Mean experience: 6.428571428571429
Normalized salary: [-0.44812908 -0.12803688  0.19205532  0.51214752         nan  1.15233192
 -2.11260852  0.83223972]


In [32]:
def clean_column(df, column, validator, fill_method="median"):
    valid_mask = validator(df[column])
    df.loc[~valid_mask, column] = pd.NA

    if fill_method == "median":
        fill_value = df[column].median()
    elif fill_method == "mean":
        fill_value = df[column].mean()
    else:
        fill_value = fill_method

    df[column] = df[column].fillna(fill_value)
    return df


df = pd.DataFrame(data)

clean_column(df, "age", lambda values: values.between(18, 80))
clean_column(df, "salary", lambda values: values > 0)
clean_column(df, "experience", lambda values: values >= 0)
clean_column(df, "city", lambda values: values.notna(), "Unknown")

df

,age,salary,experience,city,purchased
0,22.0,25000.0,1.0,Bangalore,0
1,25.0,30000.0,2.0,Mumbai,1
2,28.0,35000.0,4.0,Delhi,1
3,28.5,40000.0,5.0,Bangalore,0
4,35.0,37500.0,10.0,Mumbai,1
5,28.5,50000.0,20.0,Unknown,1
6,29.0,37500.0,3.0,Delhi,0
7,31.0,45000.0,4.0,Bangalore,1
